# Predicting Smartphone Addiction: A Validation-First Exact-Value Pipeline

This is the public Kaggle edition of the project's clearest fully original model.
It combines interpretable behavioral features, frequency encoding, fold-safe
exact-value target encoding, and multi-seed XGBoost.

The notebook is designed to run in Kaggle or Google Colab. On Kaggle, attach the
competition data and the public source dataset. On Colab, store the Kaggle token in
a secret named `KAGGLE_API_TOKEN`; never paste a token into the notebook.

The goal is reproducible learning, not a leaderboard guarantee. All target-derived
features are fitted inside the current training fold.


## What is different from the baseline

The earlier baseline used conventional CatBoost, LightGBM, and XGBoost features. This version concentrates the compute budget on the strongest reproducible signal found during public experimentation:

- exact-value target encoding for all twelve original features;
- inner cross-fitting inside every outer validation fold to prevent target leakage;
- exact-value frequency encoding from the combined train and test feature distribution;
- selected ratios and balance features;
- reference-distribution features from the CC0 original dataset;
- multi-seed XGBoost training on a Tesla T4;
- out-of-fold model selection using ROC AUC;
- automatic checkpoints, integrity checks, and submission export.

This is still an experiment, not a ranking guarantee. Public leaderboard blends above 0.971 often combine dozens of previously generated prediction files. This notebook instead keeps the training pipeline understandable and defensible.

## 1. Environment setup

Use a **T4 GPU** runtime in Colab, then run the notebook from the beginning. The installation cell may take one or two minutes.

In [ ]:
%pip -q install "kaggle>=1.7" "xgboost>=3.0,<4" "scikit-learn>=1.4,<1.9" "pandas>=2.0,<3"


In [ ]:
import gc
import json
import os
import subprocess
import sys
import time
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

COMPETITION = "playground-series-s6e8"
ORIGINAL_DATASET = "jayjoshi37/smartphone-usage-and-addiction-prediction"
TARGET = "addicted_label"
ID_COL = "id"

# Full mode is the recommended final run. Quick mode only checks the pipeline.
FULL_RUN = True
SEEDS = [42, 2026] if FULL_RUN else [42]
N_SPLITS = 5 if FULL_RUN else 3
TE_INNER_FOLDS = 5 if FULL_RUN else 3
MAX_ESTIMATORS = 5000 if FULL_RUN else 1400
EARLY_STOPPING_ROUNDS = 200 if FULL_RUN else 100

RAW_NUMERIC = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time",
]
RAW_CATEGORICAL = [
    "gender",
    "stress_level",
    "academic_work_impact",
]
RAW_COLUMNS = RAW_NUMERIC + RAW_CATEGORICAL

def detect_gpu():
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            check=True,
            capture_output=True,
            text=True,
        )
        return True, result.stdout.strip()
    except Exception:
        return False, "No NVIDIA GPU detected"

GPU_AVAILABLE, GPU_DESCRIPTION = detect_gpu()

print("Run configuration")
print(f"  Full run:        {FULL_RUN}")
print(f"  Seeds:           {SEEDS}")
print(f"  Outer folds:     {N_SPLITS}")
print(f"  Inner TE folds:  {TE_INNER_FOLDS}")
print(f"  GPU available:   {GPU_AVAILABLE}")
print(f"  GPU:             {GPU_DESCRIPTION}")

## 2. Secure Kaggle authentication

In Colab, open the **Secrets** panel, create a secret named **KAGGLE_API_TOKEN**, paste your Kaggle API token as its value, and enable notebook access. The token is read without being printed.

In [ ]:
IN_KAGGLE = Path("/kaggle/input").exists()
IN_COLAB = "google.colab" in sys.modules
KAGGLE_TOKEN_READY = False

if IN_COLAB:
    from google.colab import userdata

    try:
        kaggle_token = userdata.get("KAGGLE_API_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Create a Colab secret named KAGGLE_API_TOKEN and enable notebook access."
        ) from exc

    if not kaggle_token:
        raise RuntimeError("The KAGGLE_API_TOKEN secret is empty.")

    os.environ["KAGGLE_API_TOKEN"] = kaggle_token
    KAGGLE_TOKEN_READY = True
    print("Colab Kaggle authentication is ready.")
elif IN_KAGGLE:
    print("Kaggle runtime detected; using attached input datasets.")
else:
    KAGGLE_TOKEN_READY = bool(os.environ.get("KAGGLE_API_TOKEN"))
    print("Local runtime detected. Kaggle token available:", KAGGLE_TOKEN_READY)


## 3. Download the competition and source data

The competition page states that S6E8 was inspired by the Smartphone Usage and Addiction dataset. We use the public CC0 version only as an external reference distribution; we do not append it blindly to the training set.

In [ ]:
if IN_KAGGLE:
    ROOT = Path("/kaggle/working/s6e8_v3")
    input_root = Path("/kaggle/input")
    competition_names = ("train.csv", "test.csv", "sample_submission.csv")
    competition_candidates = [input_root / COMPETITION]
    competition_candidates.extend(
        path.parent for path in input_root.rglob("train.csv")
        if path.parent not in competition_candidates
    )
    COMP_DIR = next(
        (
            path for path in competition_candidates
            if all((path / name).exists() for name in competition_names)
        ),
        input_root / COMPETITION,
    )
    ORIG_DIR = Path("/kaggle/input/smartphone-usage-and-addiction-prediction")
else:
    ROOT = Path("/content/s6e8_v3") if IN_COLAB else Path("./s6e8_v3")
    COMP_DIR = ROOT / "competition"
    ORIG_DIR = ROOT / "original"

ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

required_competition_files = [
    COMP_DIR / "train.csv",
    COMP_DIR / "test.csv",
    COMP_DIR / "sample_submission.csv",
]

if not all(path.exists() for path in required_competition_files):
    if not KAGGLE_TOKEN_READY:
        raise FileNotFoundError(
            "Competition files are missing. On Kaggle, attach the competition data; "
            "on Colab/local, configure KAGGLE_API_TOKEN."
        )
    COMP_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["kaggle", "competitions", "download", "-c", COMPETITION,
         "-p", str(COMP_DIR)],
        check=True,
    )
    for archive_path in COMP_DIR.glob("*.zip"):
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(COMP_DIR)

original_csvs = list(ORIG_DIR.rglob("*.csv")) if ORIG_DIR.exists() else []
if not original_csvs:
    if not KAGGLE_TOKEN_READY:
        raise FileNotFoundError(
            "The source dataset is missing. Attach "
            "jayjoshi37/smartphone-usage-and-addiction-prediction on Kaggle."
        )
    ORIG_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", ORIGINAL_DATASET,
         "-p", str(ORIG_DIR), "--unzip"],
        check=True,
    )
    original_csvs = list(ORIG_DIR.rglob("*.csv"))

missing = [path.name for path in required_competition_files if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing competition files: {missing}")
if not original_csvs:
    raise FileNotFoundError("The original reference CSV was not found.")

ORIGINAL_CSV = original_csvs[0]
print("Competition data:", COMP_DIR)
print("Original data:   ", ORIGINAL_CSV)
print("Artifacts:       ", ARTIFACT_DIR)


In [ ]:
train = pd.read_csv(COMP_DIR / "train.csv")
test = pd.read_csv(COMP_DIR / "test.csv")
sample_submission = pd.read_csv(COMP_DIR / "sample_submission.csv")
original = pd.read_csv(ORIGINAL_CSV)

required_train_columns = set(RAW_COLUMNS + [ID_COL, TARGET])
required_test_columns = set(RAW_COLUMNS + [ID_COL])

assert required_train_columns.issubset(train.columns)
assert required_test_columns.issubset(test.columns)
assert set(RAW_COLUMNS + [TARGET]).issubset(original.columns)
assert list(sample_submission.columns) == [ID_COL, TARGET]

print(f"Train:       {train.shape}")
print(f"Test:        {test.shape}")
print(f"Original:    {original.shape}")
print(f"Target mean: {train[TARGET].mean():.6f}")
display(train.head())

## 4. Compact static features

Large collections of arbitrary interactions often add noise. The following ratios describe meaningful behavioral balances and remain easy to explain.

In [ ]:
def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    return numerator / denominator

def add_static_features(frame):
    result = frame[RAW_COLUMNS].copy()

    result["social_share_of_screen"] = safe_divide(
        result["social_media_hours"], result["daily_screen_time_hours"]
    )
    result["gaming_share_of_screen"] = safe_divide(
        result["gaming_hours"], result["daily_screen_time_hours"]
    )
    result["work_study_share_of_screen"] = safe_divide(
        result["work_study_hours"], result["daily_screen_time_hours"]
    )
    result["screen_to_sleep_ratio"] = safe_divide(
        result["daily_screen_time_hours"], result["sleep_hours"]
    )
    result["social_to_sleep_ratio"] = safe_divide(
        result["social_media_hours"], result["sleep_hours"]
    )
    result["social_to_work_ratio"] = safe_divide(
        result["social_media_hours"], result["work_study_hours"]
    )
    result["screen_to_work_ratio"] = safe_divide(
        result["daily_screen_time_hours"], result["work_study_hours"]
    )
    result["weekend_screen_to_sleep_ratio"] = safe_divide(
        result["weekend_screen_time"], result["sleep_hours"]
    )
    result["screen_minus_work"] = (
        result["daily_screen_time_hours"] - result["work_study_hours"]
    )
    result["screen_share_of_awake"] = safe_divide(
        result["daily_screen_time_hours"], 24.0 - result["sleep_hours"]
    )
    result["missing_count"] = frame[RAW_COLUMNS].isna().sum(axis=1)

    return result

train_features = add_static_features(train)
test_features = add_static_features(test)

print("Static feature count:", train_features.shape[1])

## 5. Exact-value frequency encoding

Frequency encoding is unsupervised. It measures how common each exact value is in the combined train and test feature distribution and therefore introduces no target leakage.

In [ ]:
combined_raw = pd.concat(
    [train[RAW_COLUMNS], test[RAW_COLUMNS]],
    axis=0,
    ignore_index=True,
)

for column in RAW_COLUMNS:
    combined_key = combined_raw[column].astype("string").fillna("__MISSING__")
    frequency = combined_key.value_counts(dropna=False) / len(combined_key)

    train_key = train[column].astype("string").fillna("__MISSING__")
    test_key = test[column].astype("string").fillna("__MISSING__")

    train_features[f"{column}__freq"] = train_key.map(frequency).astype("float32")
    test_features[f"{column}__freq"] = test_key.map(frequency).astype("float32")

print("Feature count after frequency encoding:", train_features.shape[1])

## 6. Reference-distribution features from the original data

The original dataset contains only 7,500 rows, but it exposes the source distribution that inspired the competition. We remove exact overlaps and then compute a small collection of stable statistics:

- empirical CDF positions;
- differences between class-conditional CDFs;
- distances to class medians;
- binned original target rates.

These statistics are fitted once on the external reference data, never on validation labels.

In [ ]:
original = original.dropna(subset=[TARGET]).copy()
original[TARGET] = original[TARGET].astype("int8")

def normalized_row_hash(frame):
    normalized = pd.DataFrame(index=frame.index)
    for column in RAW_NUMERIC:
        normalized[column] = (
            pd.to_numeric(frame[column], errors="coerce")
            .round(8)
            .fillna(-999999.0)
        )
    for column in RAW_CATEGORICAL:
        normalized[column] = frame[column].astype("string").fillna("__MISSING__")
    return pd.util.hash_pandas_object(normalized[RAW_COLUMNS], index=False)

train_hashes = set(normalized_row_hash(train).to_numpy())
original_hashes = normalized_row_hash(original)
original = original.loc[~original_hashes.isin(train_hashes)].copy()
original = original.drop_duplicates(subset=RAW_COLUMNS).reset_index(drop=True)

print(f"Clean original reference rows: {len(original):,}")

In [ ]:
ORIG_CDF_COLUMNS = [
    "daily_screen_time_hours",
    "weekend_screen_time",
    "social_media_hours",
]
ORIG_CLASS_COLUMNS = [
    "daily_screen_time_hours",
    "weekend_screen_time",
    "social_media_hours",
    "notifications_per_day",
    "app_opens_per_day",
]
ORIG_MEDIAN_COLUMNS = ORIG_CLASS_COLUMNS
ORIG_MEAN_COLUMNS = [
    "daily_screen_time_hours",
    "weekend_screen_time",
    "notifications_per_day",
    "app_opens_per_day",
]

def finite_values(series):
    values = pd.to_numeric(series, errors="coerce").to_numpy(dtype="float64")
    return values[np.isfinite(values)]

def empirical_cdf(values, sorted_reference):
    values = np.asarray(values, dtype="float64")
    result = np.full(len(values), np.nan, dtype="float64")
    valid = np.isfinite(values)
    result[valid] = (
        np.searchsorted(sorted_reference, values[valid], side="right")
        / len(sorted_reference)
    )
    return result

def quantile_edges(values, bins=20):
    values = values[np.isfinite(values)]
    edges = np.unique(np.quantile(values, np.linspace(0, 1, bins + 1)))
    if len(edges) < 2:
        return np.array([-np.inf, np.inf])
    edges[0], edges[-1] = -np.inf, np.inf
    return edges

orig_y = original[TARGET].to_numpy(dtype="int8")
orig_global_mean = float(orig_y.mean())
orig_references = {"cdf": {}, "class_cdf": {}, "median": {}, "mean": {}}

for column in ORIG_CDF_COLUMNS:
    orig_references["cdf"][column] = np.sort(finite_values(original[column]))

for column in ORIG_CLASS_COLUMNS:
    values = pd.to_numeric(original[column], errors="coerce").to_numpy(dtype="float64")
    orig_references["class_cdf"][column] = {
        0: np.sort(values[(orig_y == 0) & np.isfinite(values)]),
        1: np.sort(values[(orig_y == 1) & np.isfinite(values)]),
    }

for column in ORIG_MEDIAN_COLUMNS:
    values = pd.to_numeric(original[column], errors="coerce").to_numpy(dtype="float64")
    orig_references["median"][column] = {
        "all": float(np.nanmedian(values)),
        0: float(np.nanmedian(values[orig_y == 0])),
        1: float(np.nanmedian(values[orig_y == 1])),
    }

for column in ORIG_MEAN_COLUMNS:
    values = pd.to_numeric(original[column], errors="coerce").to_numpy(dtype="float64")
    edges = quantile_edges(values, bins=20)
    bin_ids = np.searchsorted(edges, values, side="right") - 1
    bin_ids = np.clip(bin_ids, 0, len(edges) - 2)
    means = np.full(len(edges) - 1, orig_global_mean, dtype="float64")
    for bin_id in range(len(means)):
        mask = (bin_ids == bin_id) & np.isfinite(values)
        if mask.any():
            means[bin_id] = float(orig_y[mask].mean())
    orig_references["mean"][column] = {"edges": edges, "means": means}

def add_original_reference_features(features, raw_frame):
    result = features.copy()

    for column in ORIG_CDF_COLUMNS:
        values = pd.to_numeric(raw_frame[column], errors="coerce").to_numpy(dtype="float64")
        result[f"{column}__orig_cdf"] = empirical_cdf(
            values, orig_references["cdf"][column]
        )

    for column in ORIG_CLASS_COLUMNS:
        values = pd.to_numeric(raw_frame[column], errors="coerce").to_numpy(dtype="float64")
        cdf_0 = empirical_cdf(values, orig_references["class_cdf"][column][0])
        cdf_1 = empirical_cdf(values, orig_references["class_cdf"][column][1])
        result[f"{column}__orig_cdf_gap"] = cdf_0 - cdf_1

    for column in ORIG_MEDIAN_COLUMNS:
        values = pd.to_numeric(raw_frame[column], errors="coerce").to_numpy(dtype="float64")
        medians = orig_references["median"][column]
        result[f"{column}__orig_median_distance"] = np.abs(values - medians["all"])
        result[f"{column}__orig_y0_median_distance"] = np.abs(values - medians[0])
        result[f"{column}__orig_y1_median_distance"] = np.abs(values - medians[1])

    for column in ORIG_MEAN_COLUMNS:
        values = pd.to_numeric(raw_frame[column], errors="coerce").to_numpy(dtype="float64")
        reference = orig_references["mean"][column]
        bin_ids = np.searchsorted(reference["edges"], values, side="right") - 1
        valid = np.isfinite(values)
        bin_ids = np.clip(bin_ids, 0, len(reference["means"]) - 1)
        encoded = np.full(len(values), orig_global_mean, dtype="float64")
        encoded[valid] = reference["means"][bin_ids[valid]]
        result[f"{column}__orig_target_mean"] = encoded

    return result

train_features = add_original_reference_features(train_features, train)
test_features = add_original_reference_features(test_features, test)

print("Feature count after original-data statistics:", train_features.shape[1])

## 7. Build the numerical XGBoost matrix

Categorical source columns are converted to stable integer codes. Exact-value target encoding is added later inside each outer fold because it depends on labels.

In [ ]:
for column in RAW_CATEGORICAL:
    combined = pd.concat(
        [train[column], test[column]],
        axis=0,
        ignore_index=True,
    ).astype("string").fillna("__MISSING__")
    categories = {value: index for index, value in enumerate(sorted(combined.unique()))}

    train_features[column] = (
        train[column].astype("string").fillna("__MISSING__").map(categories)
    )
    test_features[column] = (
        test[column].astype("string").fillna("__MISSING__").map(categories)
    )

X_base = train_features.astype("float32").to_numpy()
X_test_base = test_features.astype("float32").to_numpy()
X_base[~np.isfinite(X_base)] = np.nan
X_test_base[~np.isfinite(X_test_base)] = np.nan

y = train[TARGET].to_numpy(dtype="int8")
test_ids = test[ID_COL].copy()

# Strings preserve each observed value as a category for target encoding.
X_exact = train[RAW_COLUMNS].astype("string").fillna("__MISSING__")
X_test_exact = test[RAW_COLUMNS].astype("string").fillna("__MISSING__")

print("Base matrix:", X_base.shape)
print("Exact-value matrix:", X_exact.shape)
print("Estimated base matrix memory: " f"{X_base.nbytes / 1024**2:.1f} MB")

## 8. Leak-free multi-seed XGBoost training

For each outer fold, the target encoder is fitted only on that fold's training rows. Its internal cross-fitting creates encodings for the outer training rows, while ordinary transform is used for validation and test rows.

This nested design is slower than global target encoding, but global encoding would leak validation labels and produce a misleading score.

In [ ]:
BASE_XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "n_estimators": MAX_ESTIMATORS,
    "learning_rate": 0.03,
    "max_depth": 7,
    "min_child_weight": 5.0,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "max_bin": 256,
    "tree_method": "hist",
    "device": "cuda" if GPU_AVAILABLE else "cpu",
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    "n_jobs": -1,
    "verbosity": 0,
}

seed_oof_predictions = []
seed_test_predictions = []
training_records = []
training_started = time.perf_counter()

for seed in SEEDS:
    print("\n" + "=" * 78)
    print(f"Seed {seed}")
    print("=" * 78)

    splitter = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed,
    )
    seed_oof = np.zeros(len(y), dtype="float64")
    seed_test = np.zeros(len(test), dtype="float64")

    for fold, (train_idx, valid_idx) in enumerate(
        splitter.split(X_base, y), start=1
    ):
        fold_started = time.perf_counter()

        encoder = TargetEncoder(
            target_type="binary",
            smooth="auto",
            cv=TE_INNER_FOLDS,
            shuffle=True,
            random_state=seed + fold,
        )

        encoded_train = encoder.fit_transform(
            X_exact.iloc[train_idx], y[train_idx]
        ).astype("float32")
        encoded_valid = encoder.transform(
            X_exact.iloc[valid_idx]
        ).astype("float32")
        encoded_test = encoder.transform(X_test_exact).astype("float32")

        X_fold_train = np.column_stack([X_base[train_idx], encoded_train])
        X_fold_valid = np.column_stack([X_base[valid_idx], encoded_valid])
        X_fold_test = np.column_stack([X_test_base, encoded_test])

        model_params = dict(BASE_XGB_PARAMS)
        model_params["random_state"] = seed + fold
        model = xgb.XGBClassifier(**model_params)

        try:
            model.fit(
                X_fold_train,
                y[train_idx],
                eval_set=[(X_fold_valid, y[valid_idx])],
                verbose=250,
            )
        except xgb.core.XGBoostError as exc:
            if model_params["device"] != "cuda":
                raise
            print("CUDA training failed; retrying this fold on CPU.")
            print("Reason:", str(exc).splitlines()[0])
            model_params["device"] = "cpu"
            model = xgb.XGBClassifier(**model_params)
            model.fit(
                X_fold_train,
                y[train_idx],
                eval_set=[(X_fold_valid, y[valid_idx])],
                verbose=250,
            )

        best_iteration = getattr(model, "best_iteration", MAX_ESTIMATORS - 1)
        valid_prediction = model.predict_proba(
            X_fold_valid,
            iteration_range=(0, best_iteration + 1),
        )[:, 1]
        test_prediction = model.predict_proba(
            X_fold_test,
            iteration_range=(0, best_iteration + 1),
        )[:, 1]

        seed_oof[valid_idx] = valid_prediction
        seed_test += test_prediction / N_SPLITS

        fold_auc = roc_auc_score(y[valid_idx], valid_prediction)
        elapsed = (time.perf_counter() - fold_started) / 60
        training_records.append({
            "seed": seed,
            "fold": fold,
            "auc": fold_auc,
            "best_iteration": int(best_iteration),
            "minutes": elapsed,
        })

        print(
            f"Seed {seed} fold {fold}: "
            f"AUC={fold_auc:.6f}, "
            f"best_iteration={best_iteration}, "
            f"time={elapsed:.1f} min"
        )

        del (
            encoder,
            encoded_train,
            encoded_valid,
            encoded_test,
            X_fold_train,
            X_fold_valid,
            X_fold_test,
            model,
            valid_prediction,
            test_prediction,
        )
        gc.collect()

    seed_auc = roc_auc_score(y, seed_oof)
    print(f"Seed {seed} OOF AUC: {seed_auc:.6f}")

    seed_oof_predictions.append(seed_oof)
    seed_test_predictions.append(seed_test)
    np.save(ARTIFACT_DIR / f"oof_seed_{seed}.npy", seed_oof)
    np.save(ARTIFACT_DIR / f"test_seed_{seed}.npy", seed_test)

total_minutes = (time.perf_counter() - training_started) / 60
print(f"\nTotal training time: {total_minutes:.1f} minutes")

## 9. Select the final multi-seed ensemble

We compare probability averaging with percentile-rank averaging. The choice is made only from out-of-fold predictions, never from the public leaderboard.

In [ ]:
def percentile_rank(values):
    values = np.asarray(values, dtype="float64")
    return (rankdata(values, method="average") - 0.5) / len(values)

seed_scores = []
for seed, prediction in zip(SEEDS, seed_oof_predictions):
    seed_scores.append({
        "method": f"seed_{seed}",
        "oof_auc": roc_auc_score(y, prediction),
    })

probability_oof = np.mean(np.vstack(seed_oof_predictions), axis=0)
probability_test = np.mean(np.vstack(seed_test_predictions), axis=0)

rank_oof = np.mean(
    np.vstack([percentile_rank(p) for p in seed_oof_predictions]),
    axis=0,
)
rank_test = np.mean(
    np.vstack([percentile_rank(p) for p in seed_test_predictions]),
    axis=0,
)

candidate_predictions = {
    "probability_average": (probability_oof, probability_test),
    "rank_average": (rank_oof, rank_test),
}

for name, (oof_prediction, _) in candidate_predictions.items():
    seed_scores.append({
        "method": name,
        "oof_auc": roc_auc_score(y, oof_prediction),
    })

score_table = pd.DataFrame(seed_scores).sort_values("oof_auc", ascending=False)
display(score_table)

final_method = score_table.iloc[0]["method"]
if final_method in candidate_predictions:
    final_oof, final_test_prediction = candidate_predictions[final_method]
else:
    best_seed = int(final_method.replace("seed_", ""))
    best_index = SEEDS.index(best_seed)
    final_oof = seed_oof_predictions[best_index]
    final_test_prediction = seed_test_predictions[best_index]

final_oof_auc = roc_auc_score(y, final_oof)
print("Selected method:", final_method)
print(f"Selected OOF AUC: {final_oof_auc:.6f}")

## 10. Build and validate the Kaggle submission

The output preserves the official identifier order and checks every prediction before the CSV is written.

In [ ]:
submission = sample_submission[[ID_COL]].copy()
submission[TARGET] = np.clip(final_test_prediction, 0.0, 1.0)

assert len(submission) == len(test)
assert submission[ID_COL].equals(sample_submission[ID_COL])
assert submission[TARGET].notna().all()
assert np.isfinite(submission[TARGET]).all()
assert submission[TARGET].between(0, 1).all()
assert submission[TARGET].nunique() > 1000

submission_path = ROOT / "submission_s6e8_exact_te_v3.csv"
submission.to_csv(submission_path, index=False)

print("Submission created successfully.")
print("Path:  ", submission_path)
print("Method:", final_method)
display(submission.head())
display(submission[TARGET].describe())

## 11. Save experiment evidence

Keep the score table, fold metrics, configuration, and out-of-fold predictions. These artifacts make the experiment reproducible and provide stronger portfolio evidence than a leaderboard screenshot alone.

In [ ]:
training_table = pd.DataFrame(training_records)
training_table.to_csv(ARTIFACT_DIR / "fold_metrics.csv", index=False)
score_table.to_csv(ARTIFACT_DIR / "model_scores.csv", index=False)
pd.DataFrame({TARGET: y, "prediction": final_oof}).to_csv(
    ARTIFACT_DIR / "final_oof_predictions.csv",
    index=False,
)

run_summary = {
    "competition": COMPETITION,
    "original_dataset": ORIGINAL_DATASET,
    "full_run": FULL_RUN,
    "seeds": SEEDS,
    "outer_folds": N_SPLITS,
    "inner_target_encoding_folds": TE_INNER_FOLDS,
    "gpu_available": GPU_AVAILABLE,
    "gpu_description": GPU_DESCRIPTION,
    "base_feature_count": int(X_base.shape[1]),
    "exact_target_encoding_feature_count": len(RAW_COLUMNS),
    "selected_method": final_method,
    "selected_oof_auc": float(final_oof_auc),
    "submission_path": str(submission_path),
}

with open(ARTIFACT_DIR / "run_summary.json", "w", encoding="utf-8") as file:
    json.dump(run_summary, file, indent=2)

print("Artifacts:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(" ", path.name)

In [ ]:
if IN_COLAB:
    from google.colab import files

    files.download(str(submission_path))
else:
    print("Submission is available at:", submission_path)


## Submission strategy

1. Submit the V2 file first if it has not been submitted yet; it is your baseline.
2. Submit this V3 file as a separate experiment.
3. Record the public score and compare it with the V3 out-of-fold score.
4. Do not tune repeatedly against the public leaderboard. It contains only about 20 percent of the test labels and can reward overfitting.
5. Keep the strongest validation-backed submissions available for final selection because the private leaderboard uses the remaining 80 percent.

### Credits

The exact-value encoding direction was informed by public S6E8 experiments from Kodai Fukuda. The emphasis on aligned out-of-fold validation and model diversity was informed by Szymon Klapinski's public OOF analysis. This notebook implements its own compact training pipeline and clearly records the external dataset used.